# Load Cleaned Tables using FastParquet

### Why `engine='fastparquet'`?
- To avoid dependency issues with PyArrow, we used the `fastparquet` engine when saving.
- Therefore, we must specify the same engine when reading these Parquet files back into the new notebook.

In [ ]:
import pandas as pd
import os

# Dictionary to hold the cleaned data
df_clean = {}

# Path to the folder where you saved the files
eda_folder = 'data/eda'

# Loop through and load all files
for file_name in os.listdir(eda_folder):
    if file_name.endswith('.parquet'):
        table_name = file_name.replace('.parquet', '')
        
        # Load with engine='fastparquet' to match the saving step
        df_clean[table_name] = pd.read_parquet(f'{eda_folder}/{file_name}', engine='fastparquet')
        print(f"Successfully loaded: {table_name}")


In [ ]:

df_clean['olist_orders_dataset'].info()

# Step 1: Univariate Distribution (Price & Freight)

### Why?
- To check for outliers and understand the spread of product prices and shipping costs.
- A small number of very expensive items can skew the averages.
- `order_items` is the source table for these metrics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Fetch the clean items table
items = df_clean['olist_order_items_dataset']

# Create two side-by-side histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(items['price'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Product Price')

sns.histplot(items['freight_value'], bins=50, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Freight Value')

plt.tight_layout()
plt.show()

# Step 2: State Aggregations (Revenue & AOV)

### Why?
- To answer the core question: "Which states are driving revenue, and do they spend more per order?"
- We calculate **Total Revenue**, **Total Orders**, and **Average Order Value (AOV)**.
- We perform a logical merge (Query) between `order_items`, `orders`, and `customers` to get the `customer_state`.
- **Important:** This is a temporary merge for visualization only. We still keep our tables separate for Power BI Modeling!

In [ ]:
# 1. Temporary Merge for this specific analysis
state_df = df_clean['olist_order_items_dataset'].merge(df_clean['olist_orders_dataset'], on='order_id')
state_df = state_df.merge(df_clean['customers'], on='customer_id')

# 2. Aggregate by State
state_agg = state_df.groupby('customer_state').agg(
    total_revenue=('price', 'sum'),
    total_orders=('order_id', 'nunique'),  # Count unique orders, not items
    total_items=('order_item_id', 'count') # Count items sold
).reset_index()

# 3. Calculate Average Order Value (AOV)
state_agg['aov'] = state_agg['total_revenue'] / state_agg['total_orders']

# 4. Sort by Revenue to see Top 10
state_agg = state_agg.sort_values('total_revenue', ascending=False)

# 5. Visualize Top 10 States by Revenue
plt.figure(figsize=(12, 6))
sns.barplot(data=state_agg.head(10), x='customer_state', y='total_revenue')
plt.title('Top 10 States by Total Revenue')
plt.xlabel('Customer State')
plt.ylabel('Total Revenue')
plt.show()

# Print the table for quick analysis
print(state_agg.head(10))

# Step 3: RFM Analysis (VIP Customers)

### Why?
- To answer: "Who are the VIP customers?" 
- We use **RFM (Recency, Frequency, Monetary)** metrics.
- **Crucial:** We must use `customer_unique_id` (the actual person), NOT `customer_id` (which is tied to an order). A person can have multiple orders.
- **Recency:** Days since their last purchase.
- **Frequency:** Total number of unique orders they made.
- **Monetary:** Total amount they spent (using `payment_value`).

In [ ]:
# 1. Temporary merge for RFM analysis
rfm_df = df_clean['olist_orders_dataset'].merge(df_clean['olist_order_payments_dataset'], on='order_id')
rfm_df = rfm_df.merge(df_clean['customers'], on='customer_id')

# 2. Reference date for Recency (the latest date in the dataset)
reference_date = rfm_df['order_purchase_timestamp'].max()

# 3. Calculate RFM metrics grouped by unique customer
rfm = rfm_df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

# 4. Look at the top 10 VIPs (Highest Monetary Value)
vip_customers = rfm.sort_values('monetary', ascending=False).head(10)
print("Top 10 VIP Customers:")
print(vip_customers)

# 5. Visualize the distribution of Monetary value (log scale to handle outliers)
plt.figure(figsize=(10, 6))
sns.histplot(rfm['monetary'], bins=50, kde=True)
plt.xscale('log')
plt.title('Distribution of Customer Lifetime Value (Monetary)')
plt.show()

# Step 4: Logistics vs. Satisfaction Correlation

### Why?
- To answer: "How does delivery delay impact `review_score`?"
- We create a new feature `delay_days` (Delivered Date - Estimated Date).
- If `delay_days` > 0, the order is late.
- This shows the direct financial and reputational impact of shipping performance.

In [ ]:
# 1. Temporary merge for logistics analysis
logistics_df = df_clean['olist_orders_dataset'].merge(df_clean['olist_order_reviews_dataset'], on='order_id')

# 2. Create the Delay Feature (Calculate in days)
# Ensure they are datetime (Parquet already preserved them, but good to verify)
logistics_df['order_purchase_timestamp'] = pd.to_datetime(logistics_df['order_purchase_timestamp'])
logistics_df['order_estimated_delivery_date'] = pd.to_datetime(logistics_df['order_estimated_delivery_date'])
logistics_df['order_delivered_customer_date'] = pd.to_datetime(logistics_df['order_delivered_customer_date'])

# Calculate actual delivery time and delay
logistics_df['delay_days'] = (logistics_df['order_delivered_customer_date'] - logistics_df['order_estimated_delivery_date']).dt.days
logistics_df['is_late'] = logistics_df['delay_days'] > 0

# 3. Compare Review Scores: Late vs. On-Time
avg_score_by_late = logistics_df.groupby('is_late')['review_score'].mean().reset_index()
print("Average Review Score for On-Time vs Late Orders:")
print(avg_score_by_late)

# 4. Visualize the impact
plt.figure(figsize=(8, 6))
sns.barplot(data=avg_score_by_late, x='is_late', y='review_score')
plt.title('Impact of Late Delivery on Review Score')
plt.xlabel('Is Order Late? (True/False)')
plt.ylabel('Average Review Score')
plt.ylim(0, 5)
plt.show()

# 5. Bonus: Top 5 States with most delays
late_orders = logistics_df[logistics_df['is_late'] == True]
late_states = late_orders.merge(df_clean['customers'], on='customer_id')['customer_state'].value_counts().head(5)
print("Top 5 States with most late deliveries:")
print(late_states)

# Step 5: Product Pareto Analysis (80/20 Rule)

### Why?
- To identify the "Revenue Drivers" (the vital few) versus the "Long Tail" (the trivial many).
- This helps prioritize inventory and marketing spend on the specific categories that generate the most profit.

### How?
- Merge `order_items` with `products` to get category names.
- Group by `product_category_name` and sum the `price`.
- Sort by total revenue descending.
- Calculate the **Cumulative Revenue Percentage**.
- Visualize using a Pareto Chart (Bar chart for revenue, Line chart for cumulative %).

In [ ]:
# 1. Temporary merge for Product analysis
prod_df = df_clean['olist_order_items_dataset'].merge(df_clean['olist_products_dataset'], on='product_id')

# 2. Calculate total revenue by category
cat_rev = prod_df.groupby('product_category_name')['price'].sum().reset_index()
cat_rev = cat_rev.sort_values('price', ascending=False).reset_index(drop=True)

# 3. Calculate Cumulative Revenue Percentage
total_revenue = cat_rev['price'].sum()
cat_rev['cum_percent'] = (cat_rev['price'].cumsum() / total_revenue) * 100

# 4. Find the 80% mark
top_80_df = cat_rev[cat_rev['cum_percent'] <= 80]
num_categories = len(cat_rev)
num_top_80 = len(top_80_df)

print(f"Total Number of Categories: {num_categories}")
print(f"Categories contributing to 80% of Revenue: {num_top_80}")
print(f"Percentage of Categories driving 80% of Revenue: {(num_top_80 / num_categories) * 100:.2f}%")

# 5. Visualize the Pareto Chart (Top 10 Categories + Cumulative Line)
plt.figure(figsize=(14, 7))

# Bar Chart
ax1 = sns.barplot(data=cat_rev.head(10), x='product_category_name', y='price')
ax1.set_title('Pareto Analysis: Top 10 Revenue Categories & Cumulative Share')
ax1.set_ylabel('Total Revenue')
ax1.set_xlabel('Product Category')
plt.xticks(rotation=45, ha='right')

# Line Chart for Cumulative Percentage (on Secondary Y-Axis)
ax2 = ax1.twinx()
ax2.plot(cat_rev.index[:10], cat_rev['cum_percent'][:10], color='red', marker='o', linewidth=2)
ax2.set_ylabel('Cumulative Revenue %', color='red')
ax2.set_ylim(0, 100)
ax2.grid(False)

plt.tight_layout()
plt.show()

# Step 5: Product Pareto Analysis - RESULTS

### Key Insight:
- **22.97% of categories drive 80% of total revenue.** 
- This is a classic "Long Tail" distribution.
- The "Vital Few" categories are: **Health & Beauty, Watches & Gifts, Bed Bath & Table, Sports & Leisure, and Computers & Accessories**.
- Even the top 10 categories only sum up to ~60% of revenue, leaving a massive tail of smaller categories that make up the remaining 20%.

### Business Action:
- **Marketing:** Focus 80% of ad spend on the top 5 categories.
- **Inventory:** Guarantee stock availability for `health_beauty` (the #1 generator) and `watches_gifts`.
- **Long Tail Strategy:** Explore the bottom categories for niche products with higher margins.

# Step 6: Sellers Analysis (Pareto & Performance)

### Why?
- To answer two questions:
  1. Do the top 20% of sellers drive 80% of revenue (like products)?
  2. Which sellers are causing the most delivery delays, and how does that hurt review scores?
- This helps identify reliable partners versus risky ones.

### Strategy:
- **Revenue:** Merge `order_items` with `sellers`, group by `seller_id`, and sum the `price`.
- **Performance:** Merge `orders` with `order_items` (to get seller) and `reviews`. Calculate `delay_days` and `is_late`.
- **Final:** Combine revenue and performance into one table, then visualize.

In [ ]:
# 1. Calculate Total Revenue by Seller (Pareto)
seller_rev = df_clean['olist_order_items_dataset'].merge(df_clean['olist_sellers_dataset'], on='seller_id')
seller_rev = seller_rev.groupby('seller_id')['price'].sum().reset_index()
seller_rev = seller_rev.sort_values('price', ascending=False).reset_index(drop=True)

total_rev = seller_rev['price'].sum()
seller_rev['cum_percent'] = seller_rev['price'].cumsum() / total_rev * 100

# Calculate 80/20 threshold
top_80_sellers = seller_rev[seller_rev['cum_percent'] <= 80]
print(f"Percentage of Sellers driving 80% of Revenue: {(len(top_80_sellers)/len(seller_rev))*100:.2f}%")
print(f"Total Sellers: {len(seller_rev)}")

# 2. Analyze Seller Performance (Reviews & Delays)
seller_logic = df_clean['olist_orders_dataset'].merge(df_clean['olist_order_items_dataset'], on='order_id') # This multiplies rows
seller_logic = seller_logic.merge(df_clean['olist_order_reviews_dataset'], on='order_id') # Merges review score

# Calculate delay
seller_logic['delay_days'] = (pd.to_datetime(seller_logic['order_delivered_customer_date']) - pd.to_datetime(seller_logic['order_estimated_delivery_date'])).dt.days
seller_logic['is_late'] = seller_logic['delay_days'] > 0

# Aggregate performance by seller
seller_perf = seller_logic.groupby('seller_id').agg(
    avg_review_score=('review_score', 'mean'),
    late_rate=('is_late', 'mean'),
    total_orders=('order_id', 'nunique')
).reset_index()

# 3. Combine Revenue with Performance
seller_analysis = seller_rev[['seller_id', 'price']].merge(seller_perf, on='seller_id')

# 4. Visualize Revenue vs. Late Rate (Top 20 Sellers by Revenue)
top_20_sellers = seller_analysis.sort_values('price', ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.scatterplot(data=top_20_sellers, x='late_rate', y='avg_review_score', size='price', sizes=(50, 500))
plt.title('Top 20 Sellers: Late Rate vs. Review Score')
plt.xlabel('Late Delivery Rate (0.0 to 1.0)')
plt.ylabel('Average Review Score')
plt.show()

# Print the worst performing sellers
print("\nWorst 5 Sellers (Highest Late Rate):")
print(seller_analysis.sort_values('late_rate', ascending=False).head(5))

# Step 7: Payment Analysis (Customer Financial Behavior)

### Why?
- To understand the payment preferences of customers.
- To answer: "Do customers prefer credit cards over boleto? How many installments do they usually choose?"
- This is crucial for forecasting cash flow (e.g., if 80% are installments, revenue is delayed) and understanding purchasing power.

### Strategy:
- Use `df_clean['payments']` table directly (it is already clean).
- Analyze `payment_type` distribution.
- Analyze `payment_installments` average and distribution.
- **Advanced:** Group by `payment_installments` to see if customers who split into more installments spend more money per order.

In [ ]:
# 1. Load the payments table
payments = df_clean['olist_order_payments_dataset']

# 2. Payment Type Distribution (Percentage)
payment_type_counts = payments['payment_type'].value_counts(normalize=True) * 100
print("Payment Type Distribution (%):")
print(payment_type_counts)

# Visualize the distribution
plt.figure(figsize=(8, 6))
sns.barplot(x=payment_type_counts.index, y=payment_type_counts.values)
plt.title('Distribution of Payment Types (%)')
plt.ylabel('Percentage (%)')
plt.show()

# 3. Analyze Payment Installments
avg_installments = payments['payment_installments'].mean()
print(f"\nAverage Number of Installments: {avg_installments:.2f}")

# Does a higher number of installments mean a higher total payment value?
installment_analysis = payments.groupby('payment_installments')['payment_value'].mean().reset_index()

# 4. Visualize Installment vs. Average Value
plt.figure(figsize=(10, 6))
sns.lineplot(data=installment_analysis, x='payment_installments', y='payment_value', marker='o')
plt.title('Average Payment Value by Number of Installments')
plt.xlabel('Number of Installments')
plt.ylabel('Average Payment Value')
plt.show()

# 5. Visualize the total Payment Value Distribution
plt.figure(figsize=(10, 6))
sns.histplot(payments['payment_value'], bins=50, kde=True)
plt.title('Distribution of Payment Values')
plt.xscale('log') # Log scale to see long tail
plt.show()

# Step 7: Payment Analysis - RESULTS

### Key Insights:
- **Credit Card Dominance (73.9%):** Customers strongly prefer installment payments. This correlates with higher Average Order Value (AOV).
- **Boleto (19.0%):** A key Brazilian payment method. It takes 1-3 days to clear, explaining some of the missing `order_approved_at` values found during the cleaning phase.
- **Voucher (5.5%) & Debit (1.4%):** Small but present. Useful for retention campaigns.
- **Not Defined (0.008%):** Negligible. Can be ignored or mapped to "Unknown".

### Business Action:
- **Cash Flow:** Because 74% of payments are on credit, cash flow is delayed. Plan inventory and marketing budgets accordingly.
- **Checkout Optimization:** Offer more installment options for high-ticket items to boost AOV.

# Step 7: Payment Analysis - RESULTS (Part 2)

### Observations on Installments vs. Value:
- The line shows a smooth, positive correlation up to ~6 installments. 
- **Insight:** Offering installments allows customers to purchase higher-value items, increasing Average Order Value (AOV).
- **Technical Note:** The chart becomes jagged/spiky after 6 installments due to low sample sizes (small counts). The averages for 10, 15, or 20 installments are heavily influenced by a few outliers (e.g., one highly expensive item).

### Observations on Payment Value Distribution:
- The distribution is highly "Right-Skewed" (Long Tail). 
- Most customers pay small amounts (under $100). 
- The Mean (average) is heavily inflated by a very small number of high-value transactions (the VIP customers found in the RFM analysis).
- **Action:** When designing marketing campaigns, treat the "long tail" customers (small spenders) differently from the "VIP" customers (big spenders on the far right).

# Step 8: Seasonality Analysis (Time Series)

### Why?
- To identify peak shopping periods (e.g., Black Friday, Christmas).
- To plan inventory and marketing budgets around the "high season" and avoid overstocking in the "low season".

### Strategy:
- Extract the `Year-Month` from `order_purchase_timestamp`.
- Group by `Year-Month` and calculate:
  1. Total number of unique orders.
  2. Total revenue (sum of `price` from `order_items`).
- Plot the trends over time.

In [ ]:
# 1. Prepare data for Order Volume Seasonality
orders = df_clean['olist_orders_dataset'].copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Group by month and count unique orders
monthly_orders = orders.groupby('year_month')['order_id'].nunique().reset_index()

# Visualize Order Volume
plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_orders, x='year_month', y='order_id', marker='o')
plt.title('Number of Orders per Month (Volume Trend)')
plt.xlabel('Year-Month')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.show()

# 2. Prepare data for Revenue Seasonality (Need to join with items for prices)
orders_items = df_clean['olist_orders_dataset'].merge(df_clean['olist_order_items_dataset'], on='order_id')
orders_items['year_month'] = orders_items['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Group by month and sum price
monthly_revenue = orders_items.groupby('year_month')['price'].sum().reset_index()

# Visualize Revenue Trend
plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_revenue, x='year_month', y='price', marker='o', color='green')
plt.title('Total Revenue per Month (Seasonality)')
plt.xlabel('Year-Month')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.show()

# Step 8: Seasonality Analysis - RESULTS

### Key Insights:
- **November 2017 (Black Friday):** A massive spike in both order volume and revenue. This is the most critical selling period.
- **January 2018:** Another strong peak, likely driven by post-holiday shopping, gift card usage, or New Year promotions.
- **Dataset Cut-off:** The sharp drop to near zero in September/October 2018 is **NOT** a decline in business. It is simply the end date of the raw dataset provided.
- **Business Action:** Plan inventory, warehouse staffing, and marketing budgets heavily around November (Q4) and January (Q1) to maximize sales and avoid logistical failures.

# Master Insights Summary

## Step 1: Univariate Distribution (Price & Freight)
- **Insight:** Both price and freight values are heavily right-skewed. The majority of orders are under $100.
- **Business Action:** Do not delete the outliers (very expensive items). They represent high-value opportunities. When calculating Average Order Value (AOV), consider using the **Median** alongside the Mean to avoid skewing by rare, expensive purchases.

## Step 2: State Aggregations (Revenue & AOV)
- **Insight:** São Paulo (SP) dominates total revenue (~$5.2M) but has the **lowest** AOV (~$126). States like Bahia (BA) and Goiás (GO) have significantly **higher** AOV (~$152 and ~$146).
- **Business Action:** SP is a "Volume Market" (many cheap orders). BA and GO are "Value Markets" (fewer, but richer orders). Target marketing campaigns for high-end products specifically to BA and GO.

## Step 3: RFM Analysis (VIP Customers)
- **Insight:** The top spenders (VIPs) have a **Frequency of 1 or 2** and very high **Recency** (300-600 days). This means the platform has extremely low customer retention; people buy once, spend a lot, and never return.
- **Business Action:** Implement aggressive customer retention strategies (email campaigns, loyalty points, voucher offers) to convert these one-time high spenders into repeat customers.

## Step 4: Logistics vs. Satisfaction Correlation
- **Insight:** Late deliveries (True) drop the average review score dramatically from **4.2 to 2.3**. This is the single biggest driver of bad reviews.
- **Business Action:** Logistics is the core of the business. Fixing delivery times will directly increase customer satisfaction. SP and RJ have the highest number of late orders (due to high volume and traffic).

## Step 5: Product Pareto Analysis (80/20 Rule)
- **Insight:** **22.97%** of product categories drive **80%** of total revenue.
- **Business Action:** Focus 80% of inventory and marketing spend on the top 5 categories: **Health & Beauty, Watches & Gifts, Bed Bath & Table, Sports & Leisure, and Computers & Accessories**. This is where the money is.

## Step 6: Sellers Analysis (Pareto & Performance)
- **Insight:** Only **17.54%** of sellers generate **80%** of revenue. The scatter plot shows that some of the **highest revenue sellers** (largest bubbles) also have **high late rates** and **low review scores** (around 3.3).
- **Business Action:** These "big but slow" sellers are damaging the platform's reputation. Contact them to improve their logistics, or reduce their search ranking to prioritize faster, higher-rated sellers.

## Step 7: Payment Analysis (Customer Financial Behavior)
- **Insight:** **73.9%** of payments are made via credit card, and 19% via Boleto. There is a strong positive correlation between installments (up to 6) and payment value.
- **Business Action:** Because so many customers use credit cards, revenue is delayed (cash flow). Offer extended installment plans on expensive items to boost AOV even further.

## Step 8: Seasonality Analysis (Time Series)
- **Insight:** There are massive spikes in **November 2017 (Black Friday)** and **January 2018** (post-holiday). The sharp drop in late 2018 is due to the **dataset cutoff date**, not a decline in business.
- **Business Action:** Plan inventory, staffing, and marketing budgets heavily for Q4 (October-November) and Q1 (January) to maximize sales and avoid logistical failures.